# Petfinder Scraper Debugger (async)

Opens a Petfinder pet-detail URL in a **visible Chrome browser** via Playwright (same engine Apify uses).
Watch what loads, inspect the DOM, try carousel clicks, see exactly what's reachable.

Uses Playwright's **async API** because Jupyter runs an asyncio loop already (sync API errors out).

**Use:** edit the URL in cell 3, run cells top-to-bottom. Browser stays open between cells until cell 9.

## 1. Install dependencies (one-time)

In [89]:
%pip install playwright beautifulsoup4
!playwright install chromium

Note: you may need to restart the kernel to use updated packages.


## 2. Pick a Petfinder URL

In [91]:
TARGET_URL = "https://www.petfinder.com/cat/sebastian-860a0015-32ee-4755-bb07-e6022e1c8ef6/ga/marietta/good-mews-animal-foundation-ga173/details/"
# ↑ paste a real URL from your last Pets pipeline run

## 3. Open the page in a visible browser

In [93]:
from playwright.async_api import async_playwright
import asyncio

p = await async_playwright().start()
browser = await p.chromium.launch(headless=False, slow_mo=300)
ctx = await browser.new_context(
    user_agent=("Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"),
    viewport={"width": 1280, "height": 900},
)
page = await ctx.new_page()
await page.goto(TARGET_URL, wait_until="domcontentloaded")
print(f"Loaded: {page.url}")
print(f"Title:  {await page.title()}")

Loaded: https://www.petfinder.com/cat/sebastian-860a0015-32ee-4755-bb07-e6022e1c8ef6/ga/marietta/good-mews-animal-foundation-ga173/details/
Title:  Sebastian - Adoptable Pet | Petfinder | Petfinder


## 4. Wait for SPA data to render — try multiple selectors

In [95]:
selectors_to_try = [
    '#pet-details-story-section',
    '#animalStory',
    '[data-test="Pet_Story_Section"]',
    '[class*="description"]',
    'main p',
    'article',
]

for selector in selectors_to_try:
    try:
        el = await page.wait_for_selector(selector, timeout=4000)
        if el:
            text = (await el.inner_text())[:500]
            if text and len(text.strip()) > 30:
                print(f"✓ '{selector}' → {len(text)} chars")
                print(f"  Preview: {text[:200]}\n")
                continue
        print(f"·  '{selector}' — element exists but text < 30 chars")
    except Exception:
        print(f"✗ '{selector}' — not found within 4s")

await asyncio.sleep(2)

✓ '#pet-details-story-section' → 500 chars
  Preview: Sebastian's Story

Ben and Sebastian came to Good Mews from a county shelter for a better chance at adoption, and these two brothers have been side by side ever since.

Ben is the calmer of the pair a

✗ '#animalStory' — not found within 4s
✗ '[data-test="Pet_Story_Section"]' — not found within 4s
✗ '[class*="description"]' — not found within 4s
·  'main p' — element exists but text < 30 chars
✗ 'article' — not found within 4s


## 5. Inspect `__NEXT_DATA__` — bio + photos in the JSON

Now also probes `animalMedia` and `animal._media` since `animal.photos` is empty.

In [97]:
import json

next_json = await page.evaluate("""() => {
    const tag = document.getElementById('__NEXT_DATA__');
    return tag ? tag.textContent : null;
}""")

if not next_json:
    print("No __NEXT_DATA__ tag found.")
else:
    nd = json.loads(next_json)
    pp = nd.get("props", {}).get("pageProps", {})
    print(f"pageProps top-level keys: {list(pp.keys())}\n")

    animal = pp.get("animal") or {}
    if animal:
        desc = animal.get("description", "") or ""
        print(f"✓ animal.description: {len(desc)} chars")
        print(f"  preview: {desc[:200]}\n")

    # Probe pageProps.animalMedia (the actual photos location, per cell 5 keys)
    am = pp.get("animalMedia") or {}
    print(f"animalMedia type: {type(am).__name__}")
    if isinstance(am, dict):
        print(f"animalMedia keys: {list(am.keys())[:20]}")
    elif isinstance(am, list):
        print(f"animalMedia is a list of {len(am)} items")
        if am and isinstance(am[0], dict):
            print(f"  first item keys: {list(am[0].keys())[:20]}")

    # Probe animal._media (alternate name)
    inner_media = animal.get("_media") or {}
    if inner_media:
        print(f"\nanimal._media type: {type(inner_media).__name__}")
        if isinstance(inner_media, dict):
            print(f"animal._media keys: {list(inner_media.keys())[:20]}")
        elif isinstance(inner_media, list):
            print(f"animal._media is a list of {len(inner_media)} items")

    # Walk both for any 'url' or 'href' fields — surface any photo URLs found
    found_urls = []
    def _walk(obj, path=""):
        if isinstance(obj, dict):
            for k, v in obj.items():
                if k in ("url", "href", "large", "full", "medium", "src") and isinstance(v, str) and v.startswith("http"):
                    found_urls.append((f"{path}.{k}", v))
                _walk(v, f"{path}.{k}")
        elif isinstance(obj, list):
            for i, x in enumerate(obj):
                _walk(x, f"{path}[{i}]")
    _walk({"animalMedia": am, "animal": animal}, "")
    photo_urls_in_json = [u for _, u in found_urls if 'cloudfront' in u or 'rdcpix' in u or '/image/' in u]
    print(f"\nPhoto URLs found anywhere in JSON: {len(photo_urls_in_json)}")
    for u in photo_urls_in_json[:8]:
        print(f"  {u}")

pageProps top-level keys: ['initialApolloState', 'menuData', 'animal', 'animalMedia', 'breeds', 'drupalBreeds', 'organization', 'organizationContact', 'animalUrl', 'error', 'envVars']

✓ animal.description: 990 chars
  preview: Ben and Sebastian came to Good Mews from a county shelter for a better chance at adoption, and these two brothers have been side by side ever since.

Ben is the calmer of the pair and tends to take a 

animalMedia type: list
animalMedia is a list of 3 items
  first item keys: ['animalId', 'mediaId', 'mediaFormat', 'mediaStatus', 's3Uri', 's3Url', 'mimeType', 'originalUrl', 'publicUrl', 'originalFilename', 'position', 'mediaUrl', 'thumbnailUrl', 'mediaIndex', '__typename']

animal._media type: list
animal._media is a list of 3 items

Photo URLs found anywhere in JSON: 0


## 6. Find and click the carousel

In [99]:
carousel_button_candidates = [
    'button[aria-label="Next"]',
    'button[aria-label*="next" i]',
    '[data-testid="next"]',
    '[class*="next" i]',
    'svg[class*="chevron-right"]',
]

all_photos = set()

async def collect_visible_photos():
    urls = await page.evaluate("""() => {
        return Array.from(document.querySelectorAll('img'))
            .map(el => el.src || el.dataset.src || '')
            .filter(s => s && /\\.(jpg|jpeg|png|webp)/i.test(s))
            .filter(s => !/(logo|favicon|sprite|icon-|avatar)/i.test(s));
    }""")
    for u in urls:
        all_photos.add(u)

await collect_visible_photos()
print(f"After initial load: {len(all_photos)} unique photo URLs")

next_btn = None
for sel in carousel_button_candidates:
    btn = page.locator(sel).first
    if await btn.count() > 0:
        print(f"Found next button via: {sel}")
        next_btn = btn
        break

if next_btn:
    for i in range(4):
        try:
            await next_btn.click(timeout=2000)
            await asyncio.sleep(0.7)
            await collect_visible_photos()
            print(f"  Click {i+1} → {len(all_photos)} unique photos so far")
        except Exception as e:
            print(f"  Click {i+1} failed: {e}")
            break

print(f"\nFinal unique photos: {len(all_photos)}")
for u in list(all_photos)[:10]:
    print(f"  {u}")

After initial load: 7 unique photo URLs
Found next button via: button[aria-label*="next" i]
  Click 1 → 7 unique photos so far
  Click 2 → 7 unique photos so far
  Click 3 failed: Locator.click: Timeout 2000ms exceeded.
Call log:
  - waiting for locator("button[aria-label*=\"next\" i]").first


Final unique photos: 7
  https://dbw3zep4prcju.cloudfront.net/animal/860a0015-32ee-4755-bb07-e6022e1c8ef6/image/100bbcf8-0f40-4dc0-9f4a-90cc7f1a6d57.jpg?versionId=vZfnRdz20WMsObHFtYCEY.zkUDAf2oSS
  https://dbw3zep4prcju.cloudfront.net/animal/860a0015-32ee-4755-bb07-e6022e1c8ef6/image/31fb9d7f-f713-4d8d-bd02-db7b3927c343.jpg?versionId=gb39AmT_0MFNA.oVBnZIhgpgolB9fcHy
  https://dbw3zep4prcju.cloudfront.net/animal/591e13cf-c767-4dfa-8f4b-2e00eb325d1c/image/8be67da2-b382-4ae8-b812-362156d46b9f.jpg
  https://dbw3zep4prcju.cloudfront.net/animal/643892e6-9bf4-40e7-bf42-2a6e91ded233/image/12778b8f-942a-4018-ba31-9e5b456e231b.jpg
  https://dbw3zep4prcju.cloudfront.net/animal/860a0015-32ee-4755-bb07-e6022

## 6b. Test the production UUID-filter logic

Pulls the pet's UUID from the URL, then filters the 7 carousel-collected photos. Should leave only Sebastian's 3.

In [101]:
import re as _re

# Same logic as Furry_Friends_Marietta.py: _extract_pet_uuid + _photo_dedupe_key
def extract_pet_uuid(pet_url: str) -> str:
    if not pet_url:
        return ""
    m = _re.search(
        r"([0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12})",
        pet_url, _re.IGNORECASE,
    )
    return m.group(1).lower() if m else ""

def dedupe_key(url: str) -> str:
    return (url or "").split("?", 1)[0]

pet_uuid = extract_pet_uuid(TARGET_URL)
print(f"Pet UUID extracted from URL: {pet_uuid or '(none)'}\n")

kept = []
seen_bases = set()
for u in all_photos:
    ul = u.lower()
    base = dedupe_key(u)
    if base in seen_bases:
        print(f"  ✗ DEDUPE   {u[:90]}")
        continue
    if pet_uuid and pet_uuid not in ul:
        print(f"  ✗ WRONG-PET {u[:90]}")
        continue
    seen_bases.add(base)
    kept.append(u)
    print(f"  ✓ KEEP     {u[:90]}")

print(f"\nFiltered: {len(kept)} of {len(all_photos)} kept (after UUID filter + dedup)")
for u in kept:
    print(f"  {u}")

Pet UUID extracted from URL: 860a0015-32ee-4755-bb07-e6022e1c8ef6

  ✓ KEEP     https://dbw3zep4prcju.cloudfront.net/animal/860a0015-32ee-4755-bb07-e6022e1c8ef6/image/100
  ✓ KEEP     https://dbw3zep4prcju.cloudfront.net/animal/860a0015-32ee-4755-bb07-e6022e1c8ef6/image/31f
  ✗ WRONG-PET https://dbw3zep4prcju.cloudfront.net/animal/591e13cf-c767-4dfa-8f4b-2e00eb325d1c/image/8be
  ✗ WRONG-PET https://dbw3zep4prcju.cloudfront.net/animal/643892e6-9bf4-40e7-bf42-2a6e91ded233/image/127
  ✗ DEDUPE   https://dbw3zep4prcju.cloudfront.net/animal/860a0015-32ee-4755-bb07-e6022e1c8ef6/image/100
  ✗ WRONG-PET https://dbw3zep4prcju.cloudfront.net/animal/5dba634b-15bb-418b-a5ca-f6f50d084ded/image/9d1
  ✓ KEEP     https://dbw3zep4prcju.cloudfront.net/animal/860a0015-32ee-4755-bb07-e6022e1c8ef6/image/76b

Filtered: 3 of 7 kept (after UUID filter + dedup)
  https://dbw3zep4prcju.cloudfront.net/animal/860a0015-32ee-4755-bb07-e6022e1c8ef6/image/100bbcf8-0f40-4dc0-9f4a-90cc7f1a6d57.jpg?versionId=vZfnRdz20WM

## 6c. Run the production parser against this HTML

Imports `parse_detail_html` from `Furry_Friends_Marietta.py` directly and runs it on the HTML the browser has now. Shows what production would extract — including the new UUID filter.

In [105]:
import sys
from pathlib import Path

# Add this notebook's directory to path so we can import the production code
code_dir = Path.cwd() if 'Pets/Code' in str(Path.cwd()) else Path('.').resolve()
if str(code_dir) not in sys.path:
    sys.path.insert(0, str(code_dir))

# Reload in case we tweak the source mid-session
import importlib
import Furry_Friends_Marietta as ff
importlib.reload(ff)

rendered_html = await page.content()
detail = ff.parse_detail_html(rendered_html, pet_url=TARGET_URL)

print("=== Production parser output ===\n")
print(f"Description: {len(detail.get('description', ''))} chars")
print(f"  preview: {detail.get('description', '')[:200]}\n")
print(f"Photos: {len(detail.get('photos', []))} found")
for u in detail.get('photos', []):
    print(f"  {u}")
print(f"\nBreed: {detail.get('breed')}")
print(f"Age:   {detail.get('age')}")
print(f"Org:   {detail.get('org_name')}")

KeyError: 'NOTION_API_KEY'

## 7. Screenshot for the record

In [ ]:
from pathlib import Path
out = Path('output') / 'petfinder_debug.png'
out.parent.mkdir(exist_ok=True, parents=True)
await page.screenshot(path=str(out), full_page=True)
print(f"Saved: {out.resolve()}")

## 8. Dump full HTML for offline grep

In [ ]:
html_out = Path('output') / 'petfinder_rendered.html'
html_out.parent.mkdir(exist_ok=True, parents=True)
html_text = await page.content()
html_out.write_text(html_text, encoding='utf-8')
print(f"Saved: {html_out.resolve()}  ({html_out.stat().st_size:,} bytes)")

for keyword in ['pet-details-story-section', 'animalStory', '"animal":', 'cloudfront', 'animalMedia']:
    count = html_text.lower().count(keyword.lower())
    print(f"  '{keyword}' appears {count}x in rendered HTML")

## 9. Close the browser

In [ ]:
await browser.close()
await p.stop()
print("Closed.")